<a href="https://colab.research.google.com/github/jpccmacedo-cmyk/CEP---MVP/blob/main/Resumo_Gerencial_CN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Resumo Gerencial CN
# AUTOMATIZADOR DE CÓPIA DE ABAS EXCEL PARA UM ARQUIVO FINAL
# Google Colab
# ============================================================

!pip install openpyxl -q

from google.colab import files
from openpyxl import load_workbook, Workbook
from copy import copy
from datetime import datetime, timedelta
import os
import re

# ============================================================
# CONFIGURAÇÕES INICIAIS
# ============================================================

# Quantidade esperada de arquivos
QTDE_ARQUIVOS_ESPERADA = 7

# Extensões aceitas
EXTENSOES_EXCEL_VALIDAS = (".xlsx", ".xlsm")

# Nome fixo prioritário da aba, caso exista
NOME_ABA_GERENCIAL = "Gerencial"

# Data de referência para nome do arquivo final: hoje menos 1 dia
data_referencia = datetime.today() - timedelta(days=1)

# Nome do arquivo final
data_arquivo_final = data_referencia.strftime("%d.%m.%Y")
NOME_ARQUIVO_FINAL = f"Resumo Gerencial CN - {data_arquivo_final}.xlsx"

# Ano usado para interpretar abas no formato DDMM
# Exemplo: aba 0106 será interpretada como 01/06 do ano atual
ANO_REFERENCIA = data_referencia.year

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_nome_aba(nome):
    """
    Limpa o nome da aba para respeitar as regras do Excel:
    - Remove caracteres inválidos: : \ / ? * [ ]
    - Limita a 31 caracteres
    - Remove espaços extras
    """

    nome = os.path.splitext(nome)[0]
    nome = re.sub(r'[:\\/?*\[\]]', '', nome)
    nome = nome.strip()

    if not nome:
        nome = "Aba"

    return nome[:31]


def gerar_nome_aba_unico(nome_base, nomes_existentes):
    """
    Gera um nome de aba único, evitando duplicidade.
    O Excel não permite duas abas com o mesmo nome.
    """

    nome_base = limpar_nome_aba(nome_base)

    if nome_base not in nomes_existentes:
        return nome_base

    contador = 1

    while True:
        sufixo = f" ({contador})"
        limite_base = 31 - len(sufixo)
        nome_tentativa = nome_base[:limite_base] + sufixo

        if nome_tentativa not in nomes_existentes:
            return nome_tentativa

        contador += 1


def normalizar_nome_aba(nome):
    """
    Normaliza o nome da aba:
    - remove espaços extras
    - transforma em minúsculas
    """

    return str(nome).strip().lower()


def interpretar_aba_como_data(nome_aba, ano_referencia):
    """
    Tenta interpretar o nome da aba como uma data.

    Regras aceitas:

    1) Aba com 1 ou 2 dígitos:
       Exemplos: 1, 2, 3, 01, 02, 31
       Interpreta como dia do mês da data de referência.
       Exemplo:
       Se o mês de referência for junho,
       '03' vira 03/06/ANO_REFERENCIA.

    2) Aba com 4 dígitos:
       Exemplos: 0101, 0205, 0106
       Interpreta como DDMM.
       Exemplo:
       '0106' vira 01/06/ANO_REFERENCIA.

    Retorna um objeto datetime se conseguir interpretar.
    Retorna None se não for uma data válida.
    """

    nome = str(nome_aba).strip()

    # Só aceita abas formadas apenas por números
    if not nome.isdigit():
        return None

    try:
        # Caso 1: nome da aba é apenas o dia
        # Exemplo: 1, 2, 03, 15
        if len(nome) in [1, 2]:
            dia = int(nome)
            mes = data_referencia.month
            ano = ano_referencia

            return datetime(ano, mes, dia)

        # Caso 2: nome da aba está no formato DDMM
        # Exemplo: 0101, 0205, 0106
        elif len(nome) == 4:
            dia = int(nome[:2])
            mes = int(nome[2:])
            ano = ano_referencia

            return datetime(ano, mes, dia)

        else:
            return None

    except ValueError:
        # Exemplo: dia 32, mês 13, 3102 etc.
        return None


def encontrar_aba_mais_recente(wb_origem):
    """
    Encontra a aba correta dentro do arquivo.

    Ordem de prioridade:
    1. Se existir aba 'Gerencial', usa ela.
    2. Caso contrário, procura abas que pareçam datas.
    3. Entre as abas de data válidas, usa a mais recente.
    """

    # --------------------------------------------------------
    # 1. Prioridade para aba 'Gerencial'
    # --------------------------------------------------------
    for nome_aba in wb_origem.sheetnames:
        if normalizar_nome_aba(nome_aba) == normalizar_nome_aba(NOME_ABA_GERENCIAL):
            return nome_aba, "Aba Gerencial encontrada"

    # --------------------------------------------------------
    # 2. Procurar abas que possam ser interpretadas como datas
    # --------------------------------------------------------
    abas_datas_validas = []

    for nome_aba in wb_origem.sheetnames:
        data_interpretada = interpretar_aba_como_data(nome_aba, ANO_REFERENCIA)

        if data_interpretada is not None:
            abas_datas_validas.append({
                "nome_aba": nome_aba,
                "data": data_interpretada
            })

    # --------------------------------------------------------
    # 3. Escolher a data mais recente
    # --------------------------------------------------------
    if abas_datas_validas:
        aba_mais_recente = max(abas_datas_validas, key=lambda item: item["data"])

        return (
            aba_mais_recente["nome_aba"],
            f"Aba de data mais recente encontrada: {aba_mais_recente['data'].strftime('%d/%m/%Y')}"
        )

    # Se nada for encontrado
    return None, "Nenhuma aba Gerencial ou aba de data válida foi encontrada"


def copiar_aba_completa_como_valores(aba_origem_valores, aba_origem_formatacao, aba_destino):
    """
    Copia a aba para o arquivo final como VALORES,
    preservando o máximo possível da estrutura visual.

    Importante:
    - aba_origem_valores vem de um workbook aberto com data_only=True.
      Isso faz com que fórmulas sejam lidas como valores calculados salvos.
    - aba_origem_formatacao vem de um workbook aberto com data_only=False.
      Isso permite copiar estilos e formatação.
    """

    # --------------------------------------------------------
    # Copiar células: valores calculados e estilos
    # --------------------------------------------------------
    for linha in aba_origem_formatacao.iter_rows():
        for celula_formatacao in linha:
            coordenada = celula_formatacao.coordinate

            celula_valor = aba_origem_valores[coordenada]
            celula_destino = aba_destino[coordenada]

            # Copia o valor calculado, não a fórmula
            celula_destino.value = celula_valor.value

            # Copia estilo e formatação
            if celula_formatacao.has_style:
                celula_destino.font = copy(celula_formatacao.font)
                celula_destino.fill = copy(celula_formatacao.fill)
                celula_destino.border = copy(celula_formatacao.border)
                celula_destino.alignment = copy(celula_formatacao.alignment)
                celula_destino.number_format = celula_formatacao.number_format
                celula_destino.protection = copy(celula_formatacao.protection)

            # Copia comentário, se houver
            if celula_formatacao.comment:
                celula_destino.comment = copy(celula_formatacao.comment)

            # Copia hyperlink, se houver
            if celula_formatacao.hyperlink:
                celula_destino._hyperlink = copy(celula_formatacao.hyperlink)

    # --------------------------------------------------------
    # Copiar larguras das colunas
    # --------------------------------------------------------
    for letra_coluna, dimensao_coluna in aba_origem_formatacao.column_dimensions.items():
        aba_destino.column_dimensions[letra_coluna].width = dimensao_coluna.width
        aba_destino.column_dimensions[letra_coluna].hidden = dimensao_coluna.hidden
        aba_destino.column_dimensions[letra_coluna].outlineLevel = dimensao_coluna.outlineLevel
        aba_destino.column_dimensions[letra_coluna].collapsed = dimensao_coluna.collapsed

    # --------------------------------------------------------
    # Copiar alturas das linhas
    # --------------------------------------------------------
    for numero_linha, dimensao_linha in aba_origem_formatacao.row_dimensions.items():
        aba_destino.row_dimensions[numero_linha].height = dimensao_linha.height
        aba_destino.row_dimensions[numero_linha].hidden = dimensao_linha.hidden
        aba_destino.row_dimensions[numero_linha].outlineLevel = dimensao_linha.outlineLevel
        aba_destino.row_dimensions[numero_linha].collapsed = dimensao_linha.collapsed

    # --------------------------------------------------------
    # Copiar células mescladas
    # --------------------------------------------------------
    for intervalo_mesclado in aba_origem_formatacao.merged_cells.ranges:
        aba_destino.merge_cells(str(intervalo_mesclado))

    # --------------------------------------------------------
    # Copiar filtro automático
    # --------------------------------------------------------
    if aba_origem_formatacao.auto_filter and aba_origem_formatacao.auto_filter.ref:
        aba_destino.auto_filter.ref = aba_origem_formatacao.auto_filter.ref

    # --------------------------------------------------------
    # Copiar congelamento de painéis
    # --------------------------------------------------------
    aba_destino.freeze_panes = aba_origem_formatacao.freeze_panes

    # --------------------------------------------------------
    # Copiar visualização básica
    # --------------------------------------------------------
    aba_destino.sheet_view.showGridLines = aba_origem_formatacao.sheet_view.showGridLines

    # --------------------------------------------------------
    # Copiar configurações básicas de impressão
    # --------------------------------------------------------
    aba_destino.page_setup.orientation = aba_origem_formatacao.page_setup.orientation
    aba_destino.page_setup.paperSize = aba_origem_formatacao.page_setup.paperSize
    aba_destino.page_setup.fitToWidth = aba_origem_formatacao.page_setup.fitToWidth
    aba_destino.page_setup.fitToHeight = aba_origem_formatacao.page_setup.fitToHeight

    aba_destino.page_margins.left = aba_origem_formatacao.page_margins.left
    aba_destino.page_margins.right = aba_origem_formatacao.page_margins.right
    aba_destino.page_margins.top = aba_origem_formatacao.page_margins.top
    aba_destino.page_margins.bottom = aba_origem_formatacao.page_margins.bottom
    aba_destino.page_margins.header = aba_origem_formatacao.page_margins.header
    aba_destino.page_margins.footer = aba_origem_formatacao.page_margins.footer

    # --------------------------------------------------------
    # Copiar área de impressão
    # --------------------------------------------------------
    if aba_origem_formatacao.print_area:
        aba_destino.print_area = aba_origem_formatacao.print_area

    # --------------------------------------------------------
    # Copiar títulos de impressão
    # --------------------------------------------------------
    aba_destino.print_title_rows = aba_origem_formatacao.print_title_rows
    aba_destino.print_title_cols = aba_origem_formatacao.print_title_cols


# ============================================================
# UPLOAD DOS ARQUIVOS
# ============================================================

print("==============================================")
print("UPLOAD DOS ARQUIVOS EXCEL")
print("==============================================")
print(f"Envie os {QTDE_ARQUIVOS_ESPERADA} arquivos Excel das fábricas.")
print("Formatos aceitos: .xlsx e .xlsm")
print()

arquivos_enviados = files.upload()

# ============================================================
# VALIDAÇÃO INICIAL
# ============================================================

if not arquivos_enviados:
    raise Exception("Nenhum arquivo foi enviado. Execute a célula novamente e faça o upload dos arquivos Excel.")

nomes_arquivos = list(arquivos_enviados.keys())

print()
print("==============================================")
print("ARQUIVOS RECEBIDOS")
print("==============================================")
print(f"Quantidade de arquivos enviados: {len(nomes_arquivos)}")

if len(nomes_arquivos) != QTDE_ARQUIVOS_ESPERADA:
    print()
    print("ATENÇÃO:")
    print(f"Foram enviados {len(nomes_arquivos)} arquivos, mas o esperado era {QTDE_ARQUIVOS_ESPERADA}.")
    print("O código continuará processando os arquivos enviados.")

print()
for nome in nomes_arquivos:
    print(f"- {nome}")

# ============================================================
# VERIFICAÇÃO DAS ABAS EXISTENTES EM CADA PLANILHA
# ============================================================

print()
print("==============================================")
print("ABAS EXISTENTES EM CADA PLANILHA")
print("==============================================")

abas_por_arquivo = {}

for nome_arquivo in nomes_arquivos:

    print()
    print(f"Arquivo: {nome_arquivo}")

    try:
        if not nome_arquivo.lower().endswith(EXTENSOES_EXCEL_VALIDAS):
            print("ERRO: arquivo não é Excel válido. Use .xlsx ou .xlsm.")
            abas_por_arquivo[nome_arquivo] = []
            continue

        wb_verificacao = load_workbook(filename=nome_arquivo, read_only=True, data_only=False)

        lista_abas = wb_verificacao.sheetnames
        abas_por_arquivo[nome_arquivo] = lista_abas

        print(f"Quantidade de abas encontradas: {len(lista_abas)}")

        for aba in lista_abas:
            print(f"- {aba}")

        aba_encontrada, motivo = encontrar_aba_mais_recente(wb_verificacao)

        if aba_encontrada:
            print(f"OK: aba que será usada neste arquivo: '{aba_encontrada}'")
            print(f"Motivo: {motivo}")
        else:
            print("ATENÇÃO: nenhuma aba Gerencial ou aba de data válida foi encontrada neste arquivo.")
            print(f"Motivo: {motivo}")

        wb_verificacao.close()

    except Exception as erro:
        print(f"ERRO ao verificar abas: {erro}")
        abas_por_arquivo[nome_arquivo] = []

# ============================================================
# CRIAÇÃO DO ARQUIVO FINAL
# ============================================================

wb_final = Workbook()

# Remove a aba padrão criada automaticamente pelo openpyxl
aba_padrao = wb_final.active
wb_final.remove(aba_padrao)

# Controles de processamento
arquivos_processados = []
arquivos_com_erro = []
abas_criadas = []

# ============================================================
# PROCESSAMENTO DOS ARQUIVOS
# ============================================================

print()
print("==============================================")
print("PROCESSAMENTO")
print("==============================================")

for nome_arquivo in nomes_arquivos:

    print()
    print(f"Processando arquivo: {nome_arquivo}")

    try:
        # ----------------------------------------------------
        # Valida se é arquivo Excel
        # ----------------------------------------------------
        if not nome_arquivo.lower().endswith(EXTENSOES_EXCEL_VALIDAS):
            raise ValueError("Arquivo ignorado: extensão inválida. Use arquivos .xlsx ou .xlsm.")

        # ----------------------------------------------------
        # Abre o arquivo duas vezes:
        #
        # 1) data_only=True:
        #    usado para copiar os RESULTADOS das fórmulas como valores.
        #
        # 2) data_only=False:
        #    usado para copiar formatação, estilos e estrutura visual.
        # ----------------------------------------------------
        wb_origem_valores = load_workbook(filename=nome_arquivo, data_only=True)
        wb_origem_formatacao = load_workbook(filename=nome_arquivo, data_only=False)

        # ----------------------------------------------------
        # Encontra automaticamente a aba correta
        # ----------------------------------------------------
        nome_aba_origem_encontrada, motivo = encontrar_aba_mais_recente(wb_origem_formatacao)

        if not nome_aba_origem_encontrada:
            abas_disponiveis = ", ".join(wb_origem_formatacao.sheetnames)

            raise ValueError(
                "Nenhuma aba Gerencial ou aba de data válida foi encontrada. "
                f"Abas disponíveis no arquivo: {abas_disponiveis}"
            )

        aba_origem_valores = wb_origem_valores[nome_aba_origem_encontrada]
        aba_origem_formatacao = wb_origem_formatacao[nome_aba_origem_encontrada]

        # ----------------------------------------------------
        # Cria nome da aba final baseado no nome do arquivo
        # ----------------------------------------------------
        nomes_existentes = wb_final.sheetnames
        nome_aba_final = gerar_nome_aba_unico(nome_arquivo, nomes_existentes)

        # ----------------------------------------------------
        # Cria nova aba no arquivo final
        # ----------------------------------------------------
        aba_destino = wb_final.create_sheet(title=nome_aba_final)

        # ----------------------------------------------------
        # Copia a aba como valores, preservando formatação
        # ----------------------------------------------------
        copiar_aba_completa_como_valores(
            aba_origem_valores=aba_origem_valores,
            aba_origem_formatacao=aba_origem_formatacao,
            aba_destino=aba_destino
        )

        # ----------------------------------------------------
        # Registra sucesso
        # ----------------------------------------------------
        arquivos_processados.append({
            "arquivo": nome_arquivo,
            "aba_origem": nome_aba_origem_encontrada,
            "motivo": motivo,
            "aba_final": nome_aba_final
        })

        abas_criadas.append(nome_aba_final)

        print(
            f"SUCESSO: aba '{nome_aba_origem_encontrada}' "
            f"copiada como valores para '{nome_aba_final}'."
        )
        print(f"Motivo da escolha: {motivo}")

        wb_origem_valores.close()
        wb_origem_formatacao.close()

    except Exception as erro:
        arquivos_com_erro.append({
            "arquivo": nome_arquivo,
            "erro": str(erro)
        })

        print(f"ERRO ao processar '{nome_arquivo}': {erro}")

# ============================================================
# VALIDAÇÃO ANTES DE SALVAR
# ============================================================

if len(wb_final.sheetnames) == 0:
    raise Exception(
        "Nenhuma aba foi criada no arquivo final. "
        "Verifique se os arquivos são Excel e se existe aba Gerencial ou aba de data válida."
    )

# ============================================================
# SALVAMENTO DO ARQUIVO FINAL
# ============================================================

print()
print("==============================================")
print("SALVANDO ARQUIVO FINAL")
print("==============================================")

try:
    wb_final.save(NOME_ARQUIVO_FINAL)
    caminho_final = os.path.abspath(NOME_ARQUIVO_FINAL)

    print("Arquivo final salvo com sucesso:")
    print(caminho_final)

except Exception as erro:
    raise Exception(f"Erro ao salvar o arquivo final: {erro}")

# ============================================================
# RESUMO DA EXECUÇÃO
# ============================================================

print()
print("==============================================")
print("RESUMO DA EXECUÇÃO")
print("==============================================")

print(f"Arquivos enviados: {len(nomes_arquivos)}")
print(f"Arquivos processados com sucesso: {len(arquivos_processados)}")
print(f"Arquivos com erro: {len(arquivos_com_erro)}")

print()
print("Arquivos processados com sucesso:")

for item in arquivos_processados:
    print(
        f"- Arquivo: {item['arquivo']} | "
        f"Aba origem: {item['aba_origem']} | "
        f"Aba final: {item['aba_final']} | "
        f"Critério: {item['motivo']}"
    )

print()
print("Abas criadas no arquivo final:")

for aba in abas_criadas:
    print(f"- {aba}")

if arquivos_com_erro:
    print()
    print("Arquivos que apresentaram erro:")

    for item in arquivos_com_erro:
        print(f"- {item['arquivo']}: {item['erro']}")

print()
print(f"Nome do arquivo final gerado: {NOME_ARQUIVO_FINAL}")

# ============================================================
# DOWNLOAD AUTOMÁTICO DO ARQUIVO FINAL
# ============================================================

print()
print("==============================================")
print("DOWNLOAD")
print("==============================================")
print("Iniciando download automático do arquivo final...")

files.download(NOME_ARQUIVO_FINAL)

<>:47: SyntaxWarning: invalid escape sequence '\ '
<>:47: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_10650/2332345963.py:47: SyntaxWarning: invalid escape sequence '\ '
  - Remove caracteres inválidos: : \ / ? * [ ]


UPLOAD DOS ARQUIVOS EXCEL
Envie os 7 arquivos Excel das fábricas.
Formatos aceitos: .xlsx e .xlsm



Saving Resumo Gerencial Cuiabá 02.06.2026.xlsx to Resumo Gerencial Cuiabá 02.06.2026.xlsx
Saving Gerencial 02.xlsx to Gerencial 02.xlsx
Saving Gerencial 02-06-2026.xlsx to Gerencial 02-06-2026.xlsx
Saving Resumo Gerencial Diário - EDE - Jun26.xlsx to Resumo Gerencial Diário - EDE - Jun26.xlsx
Saving Resumo Gerencial.xlsx to Resumo Gerencial.xlsx
Saving Gerencial_02.xlsx to Gerencial_02.xlsx
Saving RG Sobradinho.xlsx to RG Sobradinho.xlsx

ARQUIVOS RECEBIDOS
Quantidade de arquivos enviados: 7

- Resumo Gerencial Cuiabá 02.06.2026.xlsx
- Gerencial 02.xlsx
- Gerencial 02-06-2026.xlsx
- Resumo Gerencial Diário - EDE - Jun26.xlsx
- Resumo Gerencial.xlsx
- Gerencial_02.xlsx
- RG Sobradinho.xlsx

ABAS EXISTENTES EM CADA PLANILHA

Arquivo: Resumo Gerencial Cuiabá 02.06.2026.xlsx
Quantidade de abas encontradas: 8
- Gerencial
- Resumo KPIs
- Estoques
- Dados_IBP
- Indicadores
- KcalKg
- Ensacadeiras
- Composição
OK: aba que será usada neste arquivo: 'Gerencial'
Motivo: Aba Gerencial encontrada



/usr/local/lib/python3.12/dist-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


SUCESSO: aba '02' copiada como valores para 'Gerencial 02'.
Motivo da escolha: Aba de data mais recente encontrada: 02/06/2026

Processando arquivo: Gerencial 02-06-2026.xlsx
SUCESSO: aba 'Gerencial' copiada como valores para 'Gerencial 02-06-2026'.
Motivo da escolha: Aba Gerencial encontrada

Processando arquivo: Resumo Gerencial Diário - EDE - Jun26.xlsx
SUCESSO: aba '2' copiada como valores para 'Resumo Gerencial Diário - EDE -'.
Motivo da escolha: Aba de data mais recente encontrada: 02/06/2026

Processando arquivo: Resumo Gerencial.xlsx
SUCESSO: aba '0206' copiada como valores para 'Resumo Gerencial'.
Motivo da escolha: Aba de data mais recente encontrada: 02/06/2026

Processando arquivo: Gerencial_02.xlsx
SUCESSO: aba 'Gerencial' copiada como valores para 'Gerencial_02'.
Motivo da escolha: Aba Gerencial encontrada

Processando arquivo: RG Sobradinho.xlsx
SUCESSO: aba '2' copiada como valores para 'RG Sobradinho'.
Motivo da escolha: Aba de data mais recente encontrada: 02/06/2026


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>